# EurecomGPT — Phase 3: MapReduce & Spark for the RAG corpus

Run in **Google Colab** (CPU runtime is fine). You build a TF-IDF index two ways — classic MapReduce (Python multiprocessing) and PySpark — then land it in Cloud Storage and BigQuery. Implement the MapReduce primitives + the two Spark stages first (`mapreduce.py`, `spark_tfidf.py`); see TASKS.md.

## 0. Setup — clone your repo and install deps

In [ ]:
# Edit the URL to YOUR repo.
!git clone https://github.com/<you>/<your-repo>.git repo 2>/dev/null || true
%cd repo
!pip -q install pyspark pyarrow google-cloud-storage google-cloud-bigquery
import sys; sys.path.insert(0, 'phase-3-mapreduce-spark')

In [ ]:
import mapreduce, corpus, spark_tfidf, report
docs = corpus.load_documents()   # deterministic: TinyShakespeare split into documents
print('documents:', len(docs))

## 1. MapReduce word count (Lecture 5)
The MAP phase is farmed across worker processes (a Hadoop stand-in); shuffle + reduce then aggregate. Explain map/shuffle/reduce in your writeup.

In [ ]:
counts = mapreduce.word_count(docs, workers=4)
top = report.top_terms(counts, n=10)
print('vocab size:', len(counts))
print('top terms:', top)

## 2. PySpark TF-IDF (Lecture 6)
The same corpus, as a chain of RDD **transformations** ending in one **action** (`collect`). Note which lines are transformations (lazy) vs actions (trigger).

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').appName('tfidf').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
docs_rdd = spark.sparkContext.parallelize(docs)
rows = spark_tfidf.build_tfidf_rows(docs_rdd, len(docs))
print('tf-idf rows:', len(rows)); print(rows[:3])

In [ ]:
spark_tfidf.save_parquet(rows, 'tfidf.parquet')
print('wrote tfidf.parquet')

## 3. Transformations, actions, lineage (writeup)
In `submission/phase3_comparison.md`, answer: which pipeline steps are **transformations** and which are **actions**? Where does Spark's **lineage** let it recover a lost partition without recomputing everything? Compare the MapReduce and Spark versions (lines of code, intermediate disk I/O, fault-tolerance story).

## 4. Upload the Parquet to Cloud Storage (public)
Colab isn't logged into gcloud — authenticate, then use the Python client (public via bucket IAM, which works with uniform bucket-level access).

In [ ]:
from google.colab import auth
auth.authenticate_user()   # opens a popup to log into your Google account

In [ ]:
PROJECT = 'REPLACE-with-your-project-id'   # <-- your Phase-0 GCP project id
BUCKET  = f'{PROJECT}-eurecomgpt'

from google.cloud import storage
client = storage.Client(project=PROJECT)
try:
    bucket = client.get_bucket(BUCKET)
except Exception:
    bucket = client.create_bucket(BUCKET, location='US')
policy = bucket.get_iam_policy(requested_policy_version=3)
policy.bindings.append({'role': 'roles/storage.objectViewer', 'members': {'allUsers'}})
bucket.set_iam_policy(policy)
bucket.blob('tfidf.parquet').upload_from_filename('tfidf.parquet')
PARQUET_URL = f'https://storage.googleapis.com/{BUCKET}/tfidf.parquet'
print('public URL:', PARQUET_URL)

## 5. Load the TF-IDF table into BigQuery and query it
The lakehouse layer: load the Parquet from GCS into BigQuery, then run a retrieval query.

In [ ]:
from google.cloud import bigquery
bq = bigquery.Client(project=PROJECT)
DATASET = f'{PROJECT}.eurecomgpt'
bq.create_dataset(bigquery.Dataset(DATASET), exists_ok=True)
TABLE = f'{DATASET}.tfidf'
job = bq.load_table_from_uri(
    f'gs://{BUCKET}/tfidf.parquet', TABLE,
    job_config=bigquery.LoadJobConfig(source_format=bigquery.SourceFormat.PARQUET,
                                      write_disposition='WRITE_TRUNCATE'))
job.result()
q = f'SELECT term, doc_id, tfidf FROM `{TABLE}` ORDER BY tfidf DESC LIMIT 10'
top_by_tfidf = [[r.term, r.doc_id, round(r.tfidf, 4)] for r in bq.query(q).result()]
print('top by tf-idf:', top_by_tfidf)

## 6. Write the submission report

In [ ]:
report.write_report(
    corpus_info={'num_docs': len(docs)},
    mapreduce={'top_terms': top, 'vocab_size': len(counts)},
    tfidf={'parquet_gcs_url': PARQUET_URL, 'num_rows': len(rows)},
    bigquery={'table': TABLE, 'top_by_tfidf': top_by_tfidf},
    comparison={'mapreduce_loc': None, 'spark_loc': None, 'notes': 'see phase3_comparison.md'},
)

Now **commit** `submission/phase3_report.json` (+ your `phase3_comparison.md`) and push.